In [1]:
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121

Looking in indexes: https://download.pytorch.org/whl/cu121

[notice] A new release of pip is available: 25.3 -> 26.0
[notice] To update, run: python3 -m pip install --upgrade pip


In [2]:
import torch
from tokenizer import Tokenizer
from model import Model

In [3]:
my_tokenizer = Tokenizer("tokenizer.json")
prompt = "Merhaba, senin adın ne söyleyebilir misin?"
tokens = my_tokenizer.encode(prompt)
torch.manual_seed(2)
model = Model(len(my_tokenizer.vocab), embedding_dim=4,device='cpu')
model(tokens)  # batch dimension ekle

tensor([[[-0.3207, -0.3656,  0.2516,  0.2175],
         [-0.2862, -0.3682,  0.2454,  0.2346],
         [-0.5852, -0.6065,  0.2935,  0.1867],
         [-0.4660, -0.4750,  0.2882,  0.2015],
         [-0.3410, -0.4257,  0.2617,  0.2349],
         [-0.4456, -0.5019,  0.2775,  0.2156],
         [-0.3842, -0.4476,  0.2475,  0.2105],
         [-0.3580, -0.4137,  0.2342,  0.2037],
         [-0.5005, -0.5247,  0.3016,  0.2089],
         [-0.3760, -0.4563,  0.2333,  0.2100],
         [-0.4571, -0.4668,  0.2659,  0.1895],
         [-0.3769, -0.4634,  0.2656,  0.2316],
         [-0.2604, -0.3194,  0.2274,  0.2173],
         [-0.4664, -0.5320,  0.3111,  0.2360],
         [-0.4472, -0.4878,  0.2890,  0.2164],
         [-0.3747, -0.4598,  0.2416,  0.2176],
         [-0.4451, -0.4670,  0.2626,  0.1938],
         [-0.3798, -0.4562,  0.2689,  0.2305],
         [-0.3969, -0.4386,  0.2935,  0.2302],
         [-0.7018, -0.6525,  0.3532,  0.1791],
         [-0.3504, -0.4242,  0.2619,  0.2301],
         [-0.

In [8]:
sentence_meaning=model(tokens)
print(sentence_meaning)

tensor([[[-0.3207, -0.3656,  0.2516,  0.2175],
         [-0.2862, -0.3682,  0.2454,  0.2346],
         [-0.5852, -0.6065,  0.2935,  0.1867],
         [-0.4660, -0.4750,  0.2882,  0.2015],
         [-0.3410, -0.4257,  0.2617,  0.2349],
         [-0.4456, -0.5019,  0.2775,  0.2156],
         [-0.3842, -0.4476,  0.2475,  0.2105],
         [-0.3580, -0.4137,  0.2342,  0.2037],
         [-0.5005, -0.5247,  0.3016,  0.2089],
         [-0.3760, -0.4563,  0.2333,  0.2100],
         [-0.4571, -0.4668,  0.2659,  0.1895],
         [-0.3769, -0.4634,  0.2656,  0.2316],
         [-0.2604, -0.3194,  0.2274,  0.2173],
         [-0.4664, -0.5320,  0.3111,  0.2360],
         [-0.4472, -0.4878,  0.2890,  0.2164],
         [-0.3747, -0.4598,  0.2416,  0.2176],
         [-0.4451, -0.4670,  0.2626,  0.1938],
         [-0.3798, -0.4562,  0.2689,  0.2305],
         [-0.3969, -0.4386,  0.2935,  0.2302],
         [-0.7018, -0.6525,  0.3532,  0.1791],
         [-0.3504, -0.4242,  0.2619,  0.2301],
         [-0.

In [7]:
all_similarities= torch.zeros((sentence_meaning.shape[0],sentence_meaning.shape[0]))

for i in range(sentence_meaning.shape[0]):
    similarities = []
    for j in range(sentence_meaning.shape[0]):
        sim = sentence_meaning[i][0]*sentence_meaning[j][0]+sentence_meaning[i][1]*sentence_meaning[j][1]+sentence_meaning[i][2]*sentence_meaning[j][2]+sentence_meaning[i][3]*sentence_meaning[j][3]
        similarities.append(sim)
    all_similarities[i] = torch.tensor(similarities)
all_similarities.detach().numpy()

ValueError: only one element tensors can be converted to Python scalars

In [16]:
sentence_meaning.squeeze(0).shape[1]

4

In [15]:
all_similarities= torch.zeros((sentence_meaning.shape[0],sentence_meaning.shape[0]))
sentence_meaning=sentence_meaning.squeeze(0)
for i in range(sentence_meaning.shape[0]):
    i_similarities = torch.zeros(sentence_meaning.shape[0])
    for j in range(sentence_meaning.shape[0]):
        for k in range(sentence_meaning.shape[1]):
            i_similarities[j] += sentence_meaning[i][k]*sentence_meaning[j][k]
    all_similarities[i] = i_similarities
all_similarities.detach().numpy()

RuntimeError: The expanded size of the tensor (1) must match the existing size (24) at non-singleton dimension 0.  Target sizes: [1].  Tensor sizes: [24]

In [11]:
sentence_meaning.shape

torch.Size([1, 24, 4])

In [20]:
# query, key, value yaklaşımıyla benzerlik hesaplama

all_similarities=sentence_meaning @ sentence_meaning.T # her bir kelime diğer kelimelerle olan benzerlikleri gösterir



In [21]:
# yukarıda her bir kelime diğer kelimelere ne kadar benziyoru gördük şimdi de her bir kelime bu cümle içinde ne kadar önemli

attention_weights = torch.softmax(all_similarities, dim=1)
attention_weights

tensor([[0.0390, 0.0387, 0.0465, 0.0427, 0.0404, 0.0429, 0.0409, 0.0399, 0.0442,
         0.0408, 0.0422, 0.0414, 0.0374, 0.0442, 0.0428, 0.0409, 0.0420, 0.0414,
         0.0416, 0.0498, 0.0404, 0.0427, 0.0392, 0.0381],
        [0.0391, 0.0389, 0.0462, 0.0427, 0.0405, 0.0429, 0.0409, 0.0399, 0.0441,
         0.0408, 0.0421, 0.0415, 0.0376, 0.0442, 0.0428, 0.0410, 0.0419, 0.0414,
         0.0416, 0.0492, 0.0405, 0.0426, 0.0394, 0.0383],
        [0.0371, 0.0365, 0.0505, 0.0435, 0.0392, 0.0437, 0.0404, 0.0387, 0.0460,
         0.0402, 0.0427, 0.0410, 0.0346, 0.0456, 0.0435, 0.0404, 0.0424, 0.0409,
         0.0411, 0.0564, 0.0393, 0.0433, 0.0371, 0.0360],
        [0.0380, 0.0375, 0.0485, 0.0432, 0.0397, 0.0433, 0.0406, 0.0393, 0.0452,
         0.0405, 0.0424, 0.0412, 0.0359, 0.0450, 0.0432, 0.0407, 0.0422, 0.0411,
         0.0414, 0.0532, 0.0399, 0.0430, 0.0381, 0.0370],
        [0.0387, 0.0384, 0.0471, 0.0428, 0.0402, 0.0430, 0.0408, 0.0397, 0.0445,
         0.0407, 0.0422, 0.0414, 0.0369

In [23]:
sentence_context_vectors = attention_weights @ sentence_meaning
sentence_context_vectors

tensor([[-0.4127, -0.4622,  0.2696,  0.2142],
        [-0.4123, -0.4619,  0.2695,  0.2143],
        [-0.4174, -0.4657,  0.2708,  0.2137],
        [-0.4151, -0.4640,  0.2702,  0.2140],
        [-0.4134, -0.4627,  0.2698,  0.2142],
        [-0.4150, -0.4639,  0.2702,  0.2140],
        [-0.4139, -0.4631,  0.2699,  0.2141],
        [-0.4134, -0.4627,  0.2698,  0.2142],
        [-0.4158, -0.4645,  0.2704,  0.2139],
        [-0.4139, -0.4631,  0.2699,  0.2141],
        [-0.4149, -0.4638,  0.2702,  0.2140],
        [-0.4140, -0.4632,  0.2700,  0.2141],
        [-0.4117, -0.4614,  0.2693,  0.2143],
        [-0.4155, -0.4643,  0.2704,  0.2140],
        [-0.4150, -0.4639,  0.2702,  0.2140],
        [-0.4139, -0.4631,  0.2699,  0.2141],
        [-0.4148, -0.4637,  0.2701,  0.2140],
        [-0.4140, -0.4632,  0.2699,  0.2141],
        [-0.4141, -0.4632,  0.2700,  0.2141],
        [-0.4191, -0.4669,  0.2713,  0.2136],
        [-0.4134, -0.4627,  0.2698,  0.2142],
        [-0.4149, -0.4639,  0.2702

In [ ]:
"""u_tokeneizer= Tokenizer("tokenizer.json")
tokens = u_tokeneizer.encode("Merhaba, senin adın ne söyleyebilir misin?")
torch.manual_seed(2)
model= Model(len(u_tokeneizer.vocab), embedding_dim=4,device='cpu')
sentence_meaning= model(tokens)

attention_weights = torch.softmax(sentence_meaning @ sentence_meaning.T, dim=1)
sentence_context_vectors = attention_weights @ sentence_meaning"""

'u_tokeneizer= Tokenizer("tokenizer.json")\ntokens = u_tokeneizer.encode("Merhaba, senin adın ne söyleyebilir misin?")\ntorch.manual_seed(2)\nmodel= Model(len(u_tokeneizer.vocab), embedding_dim=4,device=\'cpu\')\nsentence_meaning= model(tokens)\n\nattention_weights = torch.softmax(sentence_meaning @ sentence_meaning.T, dim=1)\nsentence_context_vectors = attention_weights @ sentence_meaning'

In [24]:
q_weights = torch.nn.Linear(4, 3,bias=False) # 4 boyutlu embeddingi 3 boyuta indiriyoruz
k_weights = torch.nn.Linear(4, 3,bias=False)
v_weights = torch.nn.Linear(4, 3,bias=False)


q_of_sentence = q_weights(sentence_meaning)
k_of_sentence = k_weights(sentence_meaning)
v_of_sentence = v_weights(sentence_meaning)

q_of_sentence.shape, k_of_sentence.shape, v_of_sentence.shape

(torch.Size([24, 3]), torch.Size([24, 3]), torch.Size([24, 3]))

In [ ]:
"""from plot_tokens import plot_tokens

u_sentences =[
  {  "words":q_of_sentence.detach().numpy(),
    "labels":my_tokenizer.tokenize(prompt),
    "color":"blue",},
    {
    "words":k_of_sentence.detach().numpy(),
    "labels":my_tokenizer.tokenize(prompt),
    "color":"red",
    },
    {
    "words":v_of_sentence.detach().numpy(),
    "labels":my_tokenizer.tokenize(prompt),
    "color":"green",
    }
]
plot_tokens(u_sentences, title="Query, Key, Value")"""

'from plot_tokens import plot_tokens\n\nu_sentences =[\n  {  "words":q_of_sentence.detach().numpy(),\n    "labels":my_tokenizer.tokenize(prompt),\n    "color":"blue",},\n    {\n    "words":k_of_sentence.detach().numpy(),\n    "labels":my_tokenizer.tokenize(prompt),\n    "color":"red",\n    },\n    {\n    "words":v_of_sentence.detach().numpy(),\n    "labels":my_tokenizer.tokenize(prompt),\n    "color":"green",\n    }\n]\nplot_tokens(u_sentences, title="Query, Key, Value")'

In [25]:
#şimdi query ile key arasında benzerlik hesaplayalım

k_of_sentence.shape[-1] # son boyut aynı olmalı

3

In [26]:
attention_scores = q_of_sentence @ k_of_sentence.T
attention_weights = torch.softmax(attention_scores / torch.sqrt(torch.tensor(k_of_sentence.shape[-1], dtype=torch.float32)), dim=1)

In [27]:
context_vector= attention_weights @ v_of_sentence
context_vector  # artık burda bir bağlam ve bilgi var.
# sözlük bilgisini embedding tarafında asıl bağlam da burda tutulur

tensor([[ 0.0733, -0.1224, -0.0184],
        [ 0.0733, -0.1224, -0.0184],
        [ 0.0733, -0.1223, -0.0184],
        [ 0.0733, -0.1223, -0.0184],
        [ 0.0733, -0.1224, -0.0184],
        [ 0.0733, -0.1224, -0.0184],
        [ 0.0733, -0.1224, -0.0184],
        [ 0.0733, -0.1224, -0.0184],
        [ 0.0733, -0.1224, -0.0184],
        [ 0.0733, -0.1224, -0.0184],
        [ 0.0733, -0.1223, -0.0184],
        [ 0.0733, -0.1224, -0.0184],
        [ 0.0733, -0.1224, -0.0184],
        [ 0.0733, -0.1224, -0.0184],
        [ 0.0733, -0.1224, -0.0184],
        [ 0.0733, -0.1224, -0.0184],
        [ 0.0733, -0.1223, -0.0184],
        [ 0.0733, -0.1224, -0.0184],
        [ 0.0733, -0.1224, -0.0184],
        [ 0.0733, -0.1223, -0.0184],
        [ 0.0733, -0.1224, -0.0184],
        [ 0.0733, -0.1224, -0.0184],
        [ 0.0733, -0.1224, -0.0184],
        [ 0.0733, -0.1224, -0.0184]], grad_fn=<MmBackward0>)

In [36]:
# causal self attention
# sadece geçmişteki kelimelere bakarak benzerlik hesaplama

mask= torch.tril(torch.ones((attention_weights.shape[0],attention_weights.shape[1])))  # alt üçgen maske oluşturma
#torch.softmax(mask*attention_weights,dim=1) #burda o yapınca softmaxte bir değere karşılık geliyor. Ama biz maskeleme yapmak istediğimizden dolayı spoftmaxten çıktısınında 0 olmasını istediğimden 0'ların hepsini - sonsuz yapıyorum

masked_attention_weights= attention_weights.masked_fill(mask==0, float('-inf'))
softmaxe_attention_weights= torch.softmax(masked_attention_weights, dim=1)
# bu aşamadan sonra bazı durumlarda baskın olabilecek hücreler olabiliyor. Bunlar olmasın diye dropout işlemiyle rastgele kapatabiliyoruz. Bu sayede eğitim aşamasında herhangi bir şeye odaklanmasının önüne geçiyoruz rastgele tahmin yapmasını sağlıyoruz.


In [52]:
dropout_rate= 0.5 # parametrelerin yüzde 50sini dropout yap
torch.manual_seed(42)
dropout= torch.nn.Dropout(dropout_rate)
dropout(softmaxe_attention_weights)

tensor([[0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.6666, 0.6667, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.5000, 0.5000, 0.0000, 0.5000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.4000, 0.0000, 0.4000, 0.4000, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000, 0.0000, 0.0000, 0.0000